# Generate Dataset Sintetis -- UMKM Bersama
**CC26-PSU328 | Data Science Learning Path**

Notebook ini menghasilkan dataset sintetis transaksi warung sembako yang realistis.
Dataset dirancang untuk mendukung seluruh learning path dalam proyek:
- **Data Science**: Cash Flow Forecasting, Anomaly Detection, BCG Matrix Clustering
- **AI/Cloud**: Inference pipeline input/output schema
- **Full Stack**: Struktur tabel database MySQL, REST API payload

Dataset mengandung masalah data yang disengaja (missing values, outlier, inkonsistensi format)
untuk keperluan latihan data wrangling sebelum masuk ke tahap pemodelan.

## 1. Install dan Import Library

In [ ]:
# Install dependensi jika belum tersedia
# !pip install faker pandas numpy

import pandas as pd
import numpy as np
import random
import uuid
import os
from datetime import datetime, timedelta, time
from faker import Faker

fake = Faker('id_ID')
np.random.seed(42)
random.seed(42)

OUTPUT_DIR = './dataset_sintetis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Library berhasil diimport.')

## 2. Konfigurasi Global

In [ ]:
# Rentang tanggal dataset
# Minimal 6 bulan agar Prophet bisa menangkap pola musiman dengan baik
START_DATE = datetime(2025, 10, 1)
END_DATE   = datetime(2026, 4, 20)

N_WARUNG   = 5

# Proporsi inject masalah data
MISSING_NOMINAL_RATE    = 0.015  # 1.5% nominal dikosongkan
MISSING_JAM_RATE        = 0.025  # 2.5% jam_transaksi dikosongkan
MISSING_METODE_RATE     = 0.010  # 1.0% metode_bayar dikosongkan
DUPLICATE_RATE          = 0.008  # 0.8% baris diduplikasi
OUTLIER_N               = 20     # jumlah baris outlier nominal
FORMAT_INCONSISTENT_RATE = 0.07  # 7% format jenis/metode tidak konsisten

print(f'Rentang data : {START_DATE.date()} s.d. {END_DATE.date()}')
print(f'Jumlah warung: {N_WARUNG}')

## 3. Master Data Warung

In [ ]:
warung_data = [
    {'id_warung': 'WRG-001', 'nama_warung': 'Toko Sembako Bu Sari',    'pemilik': 'Sari Wulandari',  'kota': 'Yogyakarta', 'kecamatan': 'Gondokusuman'},
    {'id_warung': 'WRG-002', 'nama_warung': 'Warung Kelontong Pak Budi','pemilik': 'Budi Santoso',    'kota': 'Sleman',     'kecamatan': 'Depok'},
    {'id_warung': 'WRG-003', 'nama_warung': 'Toko Mbak Rina',          'pemilik': 'Rina Astuti',     'kota': 'Bantul',     'kecamatan': 'Kasihan'},
    {'id_warung': 'WRG-004', 'nama_warung': 'Sembako Mas Joko',        'pemilik': 'Joko Purnomo',    'kota': 'Kulon Progo','kecamatan': 'Wates'},
    {'id_warung': 'WRG-005', 'nama_warung': 'Toko Hendra Jaya',        'pemilik': 'Hendra Wijaya',   'kota': 'Gunungkidul','kecamatan': 'Wonosari'},
]

for w in warung_data:
    w['tanggal_daftar'] = (START_DATE - timedelta(days=random.randint(30, 90))).strftime('%Y-%m-%d')
    w['status']         = 'Aktif'

df_warung = pd.DataFrame(warung_data)
df_warung

## 4. Master Data Produk

In [ ]:
# Produk dirancang realistis untuk warung sembako/kelontong
# Margin sengaja dibuat bervariasi untuk mendukung BCG Matrix clustering
produk_master = [
    # Kategori Sembako Pokok -- margin tipis, volume tinggi
    {'nama_produk': 'Beras 5kg',             'harga_jual': 75000,  'harga_pokok': 65000,  'kategori_produk': 'Sembako'},
    {'nama_produk': 'Beras 10kg',            'harga_jual': 145000, 'harga_pokok': 126000, 'kategori_produk': 'Sembako'},
    {'nama_produk': 'Minyak Goreng 1L',      'harga_jual': 18000,  'harga_pokok': 15500,  'kategori_produk': 'Sembako'},
    {'nama_produk': 'Minyak Goreng 2L',      'harga_jual': 34000,  'harga_pokok': 29500,  'kategori_produk': 'Sembako'},
    {'nama_produk': 'Gula Pasir 1kg',        'harga_jual': 17000,  'harga_pokok': 14500,  'kategori_produk': 'Sembako'},
    {'nama_produk': 'Tepung Terigu 1kg',     'harga_jual': 12000,  'harga_pokok': 10000,  'kategori_produk': 'Sembako'},
    {'nama_produk': 'Telur 1kg',             'harga_jual': 28000,  'harga_pokok': 24500,  'kategori_produk': 'Sembako'},
    {'nama_produk': 'Telur 1/4kg',           'harga_jual': 8000,   'harga_pokok': 6800,   'kategori_produk': 'Sembako'},
    {'nama_produk': 'Garam 250gr',           'harga_jual': 3500,   'harga_pokok': 2000,   'kategori_produk': 'Sembako'},
    # Kategori Mie & Snack -- margin sedang, volume sangat tinggi
    {'nama_produk': 'Indomie Goreng',        'harga_jual': 3500,   'harga_pokok': 2700,   'kategori_produk': 'Mie & Snack'},
    {'nama_produk': 'Indomie Kuah',          'harga_jual': 3500,   'harga_pokok': 2700,   'kategori_produk': 'Mie & Snack'},
    {'nama_produk': 'Mie Sedaap Goreng',     'harga_jual': 3500,   'harga_pokok': 2600,   'kategori_produk': 'Mie & Snack'},
    {'nama_produk': 'Chitato 68gr',          'harga_jual': 10000,  'harga_pokok': 7800,   'kategori_produk': 'Mie & Snack'},
    {'nama_produk': 'Oreo 137gr',            'harga_jual': 12000,  'harga_pokok': 9500,   'kategori_produk': 'Mie & Snack'},
    {'nama_produk': 'Supermi 1 dus (40pcs)', 'harga_jual': 95000,  'harga_pokok': 78000,  'kategori_produk': 'Mie & Snack'},
    # Kategori Minuman -- margin sedang
    {'nama_produk': 'Aqua 600ml',            'harga_jual': 4000,   'harga_pokok': 2800,   'kategori_produk': 'Minuman'},
    {'nama_produk': 'Aqua 1500ml',           'harga_jual': 7000,   'harga_pokok': 5000,   'kategori_produk': 'Minuman'},
    {'nama_produk': 'Teh Botol 350ml',       'harga_jual': 5000,   'harga_pokok': 3500,   'kategori_produk': 'Minuman'},
    {'nama_produk': 'Pocari Sweat 500ml',    'harga_jual': 9000,   'harga_pokok': 7000,   'kategori_produk': 'Minuman'},
    {'nama_produk': 'Kopi Kapal Api Sachet', 'harga_jual': 2000,   'harga_pokok': 1400,   'kategori_produk': 'Minuman'},
    {'nama_produk': 'Susu Kental Manis 385g','harga_jual': 14000,  'harga_pokok': 11500,  'kategori_produk': 'Minuman'},
    # Kategori Kebersihan -- margin lebih tinggi
    {'nama_produk': 'Sabun Lifebuoy 85gr',   'harga_jual': 5000,   'harga_pokok': 3500,   'kategori_produk': 'Kebersihan'},
    {'nama_produk': 'Shampo Sunsilk Sachet', 'harga_jual': 1000,   'harga_pokok': 650,    'kategori_produk': 'Kebersihan'},
    {'nama_produk': 'Rinso 800gr',           'harga_jual': 22000,  'harga_pokok': 17500,  'kategori_produk': 'Kebersihan'},
    {'nama_produk': 'Sunlight 200ml',        'harga_jual': 8000,   'harga_pokok': 6000,   'kategori_produk': 'Kebersihan'},
    {'nama_produk': 'Tisu Paseo 250 lembar', 'harga_jual': 8000,   'harga_pokok': 6000,   'kategori_produk': 'Kebersihan'},
    {'nama_produk': 'Pembalut Laurier',      'harga_jual': 15000,  'harga_pokok': 11000,  'kategori_produk': 'Kebersihan'},
    # Kategori Gas & Rokok -- margin sangat tipis tapi volume tinggi
    {'nama_produk': 'Gas LPG 3kg',           'harga_jual': 22000,  'harga_pokok': 19000,  'kategori_produk': 'Gas & Energi'},
    {'nama_produk': 'Rokok Gudang Garam 12', 'harga_jual': 25000,  'harga_pokok': 22500,  'kategori_produk': 'Rokok'},
    {'nama_produk': 'Rokok Sampoerna 16',    'harga_jual': 28000,  'harga_pokok': 25500,  'kategori_produk': 'Rokok'},
    {'nama_produk': 'Rokok Surya 16',        'harga_jual': 22000,  'harga_pokok': 19800,  'kategori_produk': 'Rokok'},
]

# Setiap warung punya subset produk yang berbeda (simulasi kondisi nyata)
produk_rows = []
for w in warung_data:
    n_produk = random.randint(18, len(produk_master))
    subset = random.sample(produk_master, k=n_produk)
    for i, p in enumerate(subset):
        produk_rows.append({
            'id_produk':       f"PRD-{w['id_warung'][-3:]}-{i+1:03d}",
            'id_warung':       w['id_warung'],
            'nama_produk':     p['nama_produk'],
            'harga_jual':      p['harga_jual'],
            'harga_pokok':     p['harga_pokok'],
            'kategori_produk': p['kategori_produk'],
            'status':          random.choices(['Aktif', 'Nonaktif'], weights=[0.92, 0.08])[0]
        })

df_produk = pd.DataFrame(produk_rows)
print(f'Total produk: {len(df_produk)} baris dari {N_WARUNG} warung')
df_produk.head(10)

## 5. Generate Transaksi Realistis

In [ ]:
# Helper: jam transaksi penjualan
# Warung sembako paling ramai pagi (belanja harian) dan sore (pulang kerja)
def jam_penjualan():
    sesi = random.choices(
        ['pagi_awal', 'pagi', 'siang', 'sore', 'malam'],
        weights=[0.15, 0.30, 0.20, 0.28, 0.07]
    )[0]
    jam_map = {
        'pagi_awal': (5, 7),
        'pagi':      (7, 10),
        'siang':     (11, 14),
        'sore':      (15, 18),
        'malam':     (18, 21),
    }
    h_min, h_max = jam_map[sesi]
    return time(random.randint(h_min, h_max - 1), random.randint(0, 59), random.randint(0, 59))

# Helper: jam pengeluaran (belanja ke distributor biasanya pagi atau siang)
def jam_pengeluaran():
    h = random.choices([6, 7, 8, 9, 13, 14], weights=[0.2, 0.25, 0.2, 0.1, 0.15, 0.1])[0]
    return time(h, random.randint(0, 59), random.randint(0, 59))

# Katalog keterangan pengeluaran per kategori
keterangan_pengeluaran = {
    'HPP': [
        'Restok beras ke agen', 'Beli minyak goreng 1 karton', 'Restok telur 1 peti',
        'Ambil indomie 2 dus ke distributor', 'Restok minuman botol', 'Beli gas LPG 3kg isi 10',
        'Restok rokok ke agen', 'Beli detergen & sabun', 'Restok snack & biskuit 1 dus',
        'Belanja ke pasar induk', 'Restok gula & tepung', 'Beli aqua 1 karton',
    ],
    'Operasional': [
        'Bensin motor untuk belanja', 'Bayar listrik warung', 'Bayar air PDAM',
        'Beli kantong kresek', 'Beli struk kasir', 'Beli label harga',
        'Ongkos kirim barang dari distributor', 'Beli selotip & alat tulis',
        'Bayar parkir pasar', 'Beli baterai timbangan',
    ],
    'Overhead': [
        'Bayar sewa toko bulan ini', 'Gaji karyawan', 'Bayar wifi bulanan',
        'Bayar iuran keamanan RT', 'Biaya perbaikan rak toko', 'Biaya tak terduga',
        'Bayar cicilan etalase', 'Biaya promosi (cetak brosur)',
    ],
}

print('Helper functions siap.')

In [ ]:
transaksi_rows = []
date_range = [START_DATE + timedelta(days=i) for i in range((END_DATE - START_DATE).days)]

for w in warung_data:
    produk_warung = df_produk[
        (df_produk['id_warung'] == w['id_warung']) &
        (df_produk['status'] == 'Aktif')
    ].to_dict('records')

    # Kelompokkan produk berdasarkan popularitas (simulasi pola jual nyata)
    # Produk mie, air mineral, rokok paling sering terjual
    produk_populer   = [p for p in produk_warung if p['kategori_produk'] in ['Mie & Snack', 'Minuman', 'Rokok']]
    produk_medium    = [p for p in produk_warung if p['kategori_produk'] in ['Sembako', 'Kebersihan']]
    produk_jarang    = [p for p in produk_warung if p['kategori_produk'] in ['Gas & Energi']]

    # Fallback jika kategori tidak ada
    if not produk_populer:  produk_populer = produk_warung
    if not produk_medium:   produk_medium  = produk_warung
    if not produk_jarang:   produk_jarang  = produk_warung

    for tanggal in date_range:
        hari         = tanggal.weekday()  # 0=Senin, 6=Minggu
        is_weekend   = hari >= 5
        is_senin     = hari == 0
        is_awal_bulan = 1 <= tanggal.day <= 7
        is_akhir_bulan = tanggal.day >= 25

        # Jumlah transaksi penjualan per hari
        # Pola realistis: awal bulan ramai, akhir bulan sepi, weekend lebih ramai
        base = random.randint(15, 35)
        if is_awal_bulan:  base = int(base * 1.4)
        if is_akhir_bulan: base = int(base * 0.75)
        if is_weekend:     base = int(base * 1.2)
        if is_senin:       base = int(base * 0.85)
        n_penjualan = base

        # Generate transaksi penjualan
        for _ in range(n_penjualan):
            # Pilih produk berdasarkan bobot popularitas
            pool = random.choices(
                ['populer', 'medium', 'jarang'],
                weights=[0.60, 0.35, 0.05]
            )[0]
            if pool == 'populer':   produk = random.choice(produk_populer)
            elif pool == 'medium':  produk = random.choice(produk_medium)
            else:                   produk = random.choice(produk_jarang)

            # Qty realistis per kategori
            if produk['kategori_produk'] in ['Mie & Snack', 'Minuman']:
                qty = random.choices([1, 2, 3, 5, 10], weights=[0.45, 0.25, 0.15, 0.10, 0.05])[0]
            elif produk['kategori_produk'] == 'Sembako':
                qty = random.choices([1, 2, 3], weights=[0.65, 0.25, 0.10])[0]
            elif produk['kategori_produk'] == 'Rokok':
                qty = random.choices([1, 2, 3], weights=[0.70, 0.20, 0.10])[0]
            else:
                qty = random.choices([1, 2], weights=[0.85, 0.15])[0]

            nominal = produk['harga_jual'] * qty

            transaksi_rows.append({
                'id_transaksi': str(uuid.uuid4())[:12].upper(),
                'id_warung':    w['id_warung'],
                'id_produk':    produk['id_produk'],
                'tanggal':      tanggal.strftime('%Y-%m-%d'),
                'jam_transaksi':str(jam_penjualan()),
                'jenis':        'Pemasukan',
                'kategori':     'Penjualan',
                'nominal':      nominal,
                'qty':          qty,
                'metode_bayar': random.choices(
                    ['Cash', 'Transfer', 'QRIS'],
                    weights=[0.55, 0.15, 0.30]
                )[0],
                'catatan': ''
            })

        # Generate transaksi pengeluaran
        # HPP: restok barang (2-4x per minggu, bukan tiap hari)
        if random.random() < 0.45 or is_senin:  # lebih sering restok di Senin
            n_hpp = random.randint(1, 3)
            for _ in range(n_hpp):
                transaksi_rows.append({
                    'id_transaksi': str(uuid.uuid4())[:12].upper(),
                    'id_warung':    w['id_warung'],
                    'id_produk':    None,
                    'tanggal':      tanggal.strftime('%Y-%m-%d'),
                    'jam_transaksi':str(jam_pengeluaran()),
                    'jenis':        'Pengeluaran',
                    'kategori':     'HPP',
                    'nominal':      random.randint(80000, 1500000),
                    'qty':          None,
                    'metode_bayar': random.choices(
                        ['Cash', 'Transfer', 'QRIS'],
                        weights=[0.60, 0.35, 0.05]
                    )[0],
                    'catatan': random.choice(keterangan_pengeluaran['HPP'])
                })

        # Operasional: 1-2x per minggu
        if random.random() < 0.25:
            transaksi_rows.append({
                'id_transaksi': str(uuid.uuid4())[:12].upper(),
                'id_warung':    w['id_warung'],
                'id_produk':    None,
                'tanggal':      tanggal.strftime('%Y-%m-%d'),
                'jam_transaksi':str(jam_pengeluaran()),
                'jenis':        'Pengeluaran',
                'kategori':     'Operasional',
                'nominal':      random.randint(5000, 120000),
                'qty':          None,
                'metode_bayar': random.choices(
                    ['Cash', 'Transfer', 'QRIS'],
                    weights=[0.75, 0.20, 0.05]
                )[0],
                'catatan': random.choice(keterangan_pengeluaran['Operasional'])
            })

        # Overhead: hanya awal bulan
        if is_awal_bulan and tanggal.day <= 3 and random.random() < 0.6:
            transaksi_rows.append({
                'id_transaksi': str(uuid.uuid4())[:12].upper(),
                'id_warung':    w['id_warung'],
                'id_produk':    None,
                'tanggal':      tanggal.strftime('%Y-%m-%d'),
                'jam_transaksi':str(jam_pengeluaran()),
                'jenis':        'Pengeluaran',
                'kategori':     'Overhead',
                'nominal':      random.randint(150000, 1000000),
                'qty':          None,
                'metode_bayar': random.choices(
                    ['Cash', 'Transfer'],
                    weights=[0.40, 0.60]
                )[0],
                'catatan': random.choice(keterangan_pengeluaran['Overhead'])
            })

df_transaksi = pd.DataFrame(transaksi_rows)
print(f'Total transaksi bersih: {len(df_transaksi):,} baris')
df_transaksi.head()

## 6. Inject Masalah Data (untuk keperluan wrangling)

In [ ]:
n = len(df_transaksi)

# ── Missing values ──────────────────────────────────────────────
df_transaksi.loc[df_transaksi.sample(frac=MISSING_NOMINAL_RATE).index, 'nominal']      = np.nan
df_transaksi.loc[df_transaksi.sample(frac=MISSING_JAM_RATE).index,    'jam_transaksi'] = None
df_transaksi.loc[df_transaksi.sample(frac=MISSING_METODE_RATE).index, 'metode_bayar']  = None

# catatan kosong untuk sebagian pengeluaran
idx_out = df_transaksi[df_transaksi['jenis'] == 'Pengeluaran'].sample(frac=0.08).index
df_transaksi.loc[idx_out, 'catatan'] = None

# ── Inkonsistensi format ──────────────────────────────────────────
# Variasi penulisan kolom jenis
jenis_typo = {
    'Pemasukan':   ['pemasukan', 'PEMASUKAN', 'Pemasokan', 'income', 'masuk'],
    'Pengeluaran': ['pengeluaran', 'PENGELUARAN', 'Pengeluaaran', 'expense', 'keluar'],
}
for original, variants in jenis_typo.items():
    idx = df_transaksi[df_transaksi['jenis'] == original].sample(frac=FORMAT_INCONSISTENT_RATE).index
    df_transaksi.loc[idx, 'jenis'] = [random.choice(variants) for _ in range(len(idx))]

# Variasi penulisan metode_bayar
metode_typo = {
    'Cash':     ['cash', 'CASH', 'tunai', 'Tunai', 'cash '],
    'Transfer': ['transfer', 'TRANSFER', 'tf', 'TF', 'Trasfer'],
    'QRIS':     ['qris', 'Qris', 'scan', 'qr'],
}
for original, variants in metode_typo.items():
    idx = df_transaksi[df_transaksi['metode_bayar'] == original].sample(frac=FORMAT_INCONSISTENT_RATE).index
    df_transaksi.loc[idx, 'metode_bayar'] = [random.choice(variants) for _ in range(len(idx))]

# Variasi format tanggal (~2%)
idx_tgl = df_transaksi.sample(frac=0.02).index
for i in idx_tgl:
    tgl = df_transaksi.loc[i, 'tanggal']
    try:
        dt  = datetime.strptime(str(tgl), '%Y-%m-%d')
        fmt = random.choice(['%d/%m/%Y', '%d-%m-%Y', '%Y/%m/%d'])
        df_transaksi.loc[i, 'tanggal'] = dt.strftime(fmt)
    except:
        pass

# ── Outlier nominal ────────────────────────────────────────────────
# Simulasi salah ketik (lupa strip nol)
idx_outlier = df_transaksi.sample(n=OUTLIER_N).index
df_transaksi.loc[idx_outlier, 'nominal'] = df_transaksi.loc[idx_outlier, 'nominal'].apply(
    lambda x: x * random.choice([10, 100]) if pd.notna(x) else x
)

# Simulasi nominal 0 atau negatif (kesalahan input)
idx_zero = df_transaksi.sample(n=10).index
df_transaksi.loc[idx_zero, 'nominal'] = random.choices([0, -5000, -15000, -25000], k=10)

# ── Duplikasi baris ────────────────────────────────────────────────
dup = df_transaksi.sample(frac=DUPLICATE_RATE)
df_transaksi = pd.concat([df_transaksi, dup], ignore_index=True)

# ── Inkonsistensi kategori vs jenis ────────────────────────────────
# Beberapa pemasukan salah dikategorikan sebagai HPP
idx_kat = df_transaksi[
    df_transaksi['jenis'].str.lower().isin(['pemasukan', 'income', 'masuk'])
].sample(frac=0.015).index
df_transaksi.loc[idx_kat, 'kategori'] = random.choices(['HPP', 'Operasional'], k=len(idx_kat))

# ── Referential integrity issue ────────────────────────────────────
# Beberapa id_produk tidak valid (simulasi bug di FE saat input)
idx_ref = df_transaksi[
    df_transaksi['jenis'].str.lower().isin(['pemasukan', 'income', 'masuk'])
].sample(frac=0.008).index
df_transaksi.loc[idx_ref, 'id_produk'] = 'PRD-INVALID-999'

# Shuffle baris agar tidak berurutan
df_transaksi = df_transaksi.sample(frac=1).reset_index(drop=True)

print(f'Total transaksi setelah inject masalah: {len(df_transaksi):,} baris')
print(f'\nMissing values per kolom:')
print(df_transaksi.isnull().sum())

## 7. Ringkasan Dataset

In [ ]:
print('==== RINGKASAN DATASET ====')
print(f'warung.csv    : {len(df_warung):,} baris')
print(f'produk.csv    : {len(df_produk):,} baris')
print(f'transaksi.csv : {len(df_transaksi):,} baris')
print()
print('==== DISTRIBUSI JENIS TRANSAKSI ====')
print(df_transaksi['jenis'].value_counts())
print()
print('==== DISTRIBUSI KATEGORI ====')
print(df_transaksi['kategori'].value_counts())
print()
print('==== STATISTIK NOMINAL ====')
print(df_transaksi['nominal'].describe())
print()
print('==== MASALAH DATA YANG DIINJEKSIKAN ====')
print('1. Missing values   : nominal, jam_transaksi, metode_bayar, catatan')
print('2. Format tidak konsisten : jenis (5 variasi), metode_bayar (5 variasi), tanggal (3 format)')
print('3. Outlier nominal  : 20 baris nominal x10 atau x100, 10 baris nominal 0 atau negatif')
print(f'4. Duplikasi        : {int(len(df_transaksi) * DUPLICATE_RATE)} baris terduplikasi')
print('5. Inkonsistensi kategori vs jenis : ~1.5% pemasukan berkategori HPP/Operasional')
print('6. Referential integrity : ~0.8% id_produk tidak valid (PRD-INVALID-999)')

## 8. Export CSV

In [ ]:
df_warung.to_csv(f'{OUTPUT_DIR}/warung.csv',       index=False, encoding='utf-8')
df_produk.to_csv(f'{OUTPUT_DIR}/produk.csv',       index=False, encoding='utf-8')
df_transaksi.to_csv(f'{OUTPUT_DIR}/transaksi.csv', index=False, encoding='utf-8')

print('Dataset berhasil disimpan:')
print(f'  {OUTPUT_DIR}/warung.csv')
print(f'  {OUTPUT_DIR}/produk.csv')
print(f'  {OUTPUT_DIR}/transaksi.csv')